In [1]:
import pandas as pd 
import numpy as np
import os
from pathlib import Path

In [2]:
# import train data 
df = pd.read_csv("/home/sjoon/projects/brain_connectivity_classifier/data/raw/PIOP2_restingstate.csv")
df.shape

(224, 26797)

In [3]:
# Step 1: Extract all connectivity columns
connectivity_cols = [col for col in df.columns if '~' in str(col)]
print(f"Total connectivity columns: {len(connectivity_cols)}")

# Step 2: Extract unique regions while preserving order
regions_ordered = []
seen_regions = set()

for col in connectivity_cols:
    region1, region2 = col.split('~')
    
    if region1 not in seen_regions:
        regions_ordered.append(region1)
        seen_regions.add(region1)
    
    if region2 not in seen_regions:
        regions_ordered.append(region2)
        seen_regions.add(region2)

print(f"Total unique regions: {len(regions_ordered)}")
print(f"First 10 regions: {regions_ordered[:10]}")

# Step 3: Create a mapping of connectivity values for quick lookup
# Format: {subject: {(region1, region2): value}}
connectivity_map = {}

for idx, row in df.iterrows():
    subject = row['subject']
    connectivity_map[subject] = {}
    
    for col in connectivity_cols:
        region1, region2 = col.split('~')
        connectivity_map[subject][(region1, region2)] = row[col]

print(f"\nConnectivity map created for {len(connectivity_map)} subjects")

# Step 4: Build the transformed dataframe
rows = []

for subject in df['subject']:
    for target_region in regions_ordered:
        # Create a row for this subject-region pair
        row_data = {
            'subject': subject,
            'region': target_region
        }
        
        # For each of the 232 regions, find connectivity value
        for source_region in regions_ordered:
            if target_region == source_region:
                # Diagonal: check if self-connection exists, otherwise assign 1
                if (target_region, source_region) in connectivity_map[subject]:
                    row_data[source_region] = connectivity_map[subject][(target_region, source_region)]
                elif (source_region, target_region) in connectivity_map[subject]:
                    row_data[source_region] = connectivity_map[subject][(source_region, target_region)]
                else:
                    row_data[source_region] = 1.0
            else:
                # Off-diagonal: check both directions
                if (target_region, source_region) in connectivity_map[subject]:
                    row_data[source_region] = connectivity_map[subject][(target_region, source_region)]
                elif (source_region, target_region) in connectivity_map[subject]:
                    row_data[source_region] = connectivity_map[subject][(source_region, target_region)]
                else:
                    row_data[source_region] = np.nan  # No connection found
        
        rows.append(row_data)

# Create the final dataframe
df_transformed = pd.DataFrame(rows)

# Ensure column order: subject, region, then all regions in order
final_columns = ['subject', 'region'] + regions_ordered
df_transformed = df_transformed[final_columns]

print(f"\n{'='*50}")
print(f"TRANSFORMATION COMPLETE")
print(f"{'='*50}")
print(f"Final shape: {df_transformed.shape}")
print(f"Expected: (51968, 234)")
print(f"\nFirst few rows:")
print(df_transformed.iloc[:5, :7])
print(f"\nColumn names (first 10):")
print(df_transformed.columns[:10].tolist())
print(f"\nChecking for missing values:")
print(f"Total NaN values: {df_transformed.isna().sum().sum()}")

Total connectivity columns: 26796
Total unique regions: 232
First 10 regions: ['LH_VisCent_ExStr_2', 'LH_VisCent_ExStr_1', 'LH_VisCent_Striate_1', 'LH_VisCent_ExStr_3', 'LH_VisCent_ExStr_4', 'LH_VisCent_ExStr_5', 'LH_VisPeri_ExStrInf_1', 'LH_VisPeri_ExStrInf_2', 'LH_VisPeri_ExStrInf_3', 'LH_VisPeri_StriCal_1']

Connectivity map created for 224 subjects

TRANSFORMATION COMPLETE
Final shape: (51968, 234)
Expected: (51968, 234)

First few rows:
       subject                region  LH_VisCent_ExStr_2  LH_VisCent_ExStr_1  \
0  sub-0001_P2    LH_VisCent_ExStr_2            1.000000            0.449238   
1  sub-0001_P2    LH_VisCent_ExStr_1            0.449238            1.000000   
2  sub-0001_P2  LH_VisCent_Striate_1            0.524210            0.730071   
3  sub-0001_P2    LH_VisCent_ExStr_3            0.276710            0.497770   
4  sub-0001_P2    LH_VisCent_ExStr_4            0.320928            0.535327   

   LH_VisCent_Striate_1  LH_VisCent_ExStr_3  LH_VisCent_ExStr_4  
0      

In [4]:
# Step 1: Extract the actual region order from df_transformed
regions_ordered = df_transformed['region'].unique().tolist()

print(f"{'='*60}")
print(f"REGION ORDER VERIFICATION")
print(f"{'='*60}")
print(f"Total unique regions in df_transformed: {len(regions_ordered)}")
print(f"Expected: 232")

print(f"\nFirst 10 regions:")
for i, reg in enumerate(regions_ordered[:10], 1):
    print(f"  {i}. {reg}")

print(f"\nLast 10 regions:")
for i, reg in enumerate(regions_ordered[-10:], len(regions_ordered)-9):
    print(f"  {i}. {reg}")

# Step 2: Verify the region order matches your provided list
expected_first_5 = ['LH_VisCent_ExStr_2', 'LH_VisCent_ExStr_1', 'LH_VisCent_Striate_1', 
                     'LH_VisCent_ExStr_3', 'LH_VisCent_ExStr_4']
expected_last_5 = ['aGP-lh', 'aPUT-lh', 'pPUT-lh', 'aCAU-lh', 'pCAU-lh']

print(f"\n{'='*60}")
print(f"ORDER VERIFICATION")
print(f"{'='*60}")
print(f"Expected first 5: {expected_first_5}")
print(f"Actual first 5:   {regions_ordered[:5]}")
print(f"Match: {regions_ordered[:5] == expected_first_5}")

print(f"\nExpected last 5: {expected_last_5}")
print(f"Actual last 5:   {regions_ordered[-5:]}")
print(f"Match: {regions_ordered[-5:] == expected_last_5}")

# Step 3: Identify left and right hemisphere regions (CORRECTED with hyphen)
left_regions = [reg for reg in regions_ordered if reg.startswith('LH_') or reg.endswith('-lh')]
right_regions = [reg for reg in regions_ordered if reg.startswith('RH_') or reg.endswith('-rh')]

# Step 4: Verify counts
print(f"\n{'='*60}")
print(f"HEMISPHERE REGION COUNTS (CORRECTED)")
print(f"{'='*60}")

# Left hemisphere breakdown
left_cortical = [reg for reg in left_regions if reg.startswith('LH_')]
left_subcortical = [reg for reg in left_regions if reg.endswith('-lh')]

print(f"\nLEFT HEMISPHERE:")
print(f"  Cortical (LH_ prefix): {len(left_cortical)}")
print(f"  Subcortical (-lh suffix with HYPHEN): {len(left_subcortical)}")
print(f"  TOTAL: {len(left_regions)}")
print(f"  Expected: 116 (100 cortical + 16 subcortical)")

# Right hemisphere breakdown
right_cortical = [reg for reg in right_regions if reg.startswith('RH_')]
right_subcortical = [reg for reg in right_regions if reg.endswith('-rh')]

print(f"\nRIGHT HEMISPHERE:")
print(f"  Cortical (RH_ prefix): {len(right_cortical)}")
print(f"  Subcortical (-rh suffix with HYPHEN): {len(right_subcortical)}")
print(f"  TOTAL: {len(right_regions)}")
print(f"  Expected: 116 (100 cortical + 16 subcortical)")

# Step 5: Verify total
print(f"\nTOTAL REGIONS: {len(left_regions) + len(right_regions)}")
print(f"Expected: 232")

# Show examples
print(f"\n{'='*60}")
print(f"EXAMPLE REGIONS")
print(f"{'='*60}")
print(f"\nFirst 5 LEFT cortical: {left_cortical[:5]}")
print(f"Last 5 LEFT subcortical: {left_subcortical[-5:]}")
print(f"\nFirst 5 RIGHT cortical: {right_cortical[:5]}")
print(f"Last 5 RIGHT subcortical: {right_subcortical[-5:]}")

# Step 6: Create left hemisphere dataset
print(f"\n{'='*60}")
print(f"CREATING LEFT HEMISPHERE DATASET")
print(f"{'='*60}")

# Filter rows: keep only left hemisphere regions
df_left = df_transformed[df_transformed['region'].isin(left_regions)].copy()

# Filter columns: keep subject, region, and only left hemisphere connectivity columns
# IMPORTANT: Maintain the order from regions_ordered
left_columns_ordered = ['subject', 'region'] + [reg for reg in regions_ordered if reg in left_regions]
df_left = df_left[left_columns_ordered]

print(f"Shape: {df_left.shape}")
print(f"Expected: (25984, 118)")
print(f"\nFirst few rows and columns:")
print(df_left.iloc[:3, :7])

# Step 7: Create right hemisphere dataset
print(f"\n{'='*60}")
print(f"CREATING RIGHT HEMISPHERE DATASET")
print(f"{'='*60}")

# Filter rows: keep only right hemisphere regions
df_right = df_transformed[df_transformed['region'].isin(right_regions)].copy()

# Filter columns: keep subject, region, and only right hemisphere connectivity columns
# IMPORTANT: Maintain the order from regions_ordered
right_columns_ordered = ['subject', 'region'] + [reg for reg in regions_ordered if reg in right_regions]
df_right = df_right[right_columns_ordered]

print(f"Shape: {df_right.shape}")
print(f"Expected: (25984, 118)")
print(f"\nFirst few rows and columns:")
print(df_right.iloc[:3, :7])

# Step 8: Final verification
print(f"\n{'='*60}")
print(f"FINAL VERIFICATION")
print(f"{'='*60}")
print(f"\nLeft hemisphere dataset:")
print(f"  Rows: {df_left.shape[0]} (expected: 25984)")
print(f"  Columns: {df_left.shape[1]} (expected: 118)")
print(f"  Unique subjects: {df_left['subject'].nunique()} (expected: 224)")
print(f"  Unique regions: {df_left['region'].nunique()} (expected: 116)")
print(f"  Column order preserved: {df_left.columns[2:7].tolist()}")

print(f"\nRight hemisphere dataset:")
print(f"  Rows: {df_right.shape[0]} (expected: 25984)")
print(f"  Columns: {df_right.shape[1]} (expected: 118)")
print(f"  Unique subjects: {df_right['subject'].nunique()} (expected: 224)")
print(f"  Unique regions: {df_right['region'].nunique()} (expected: 116)")
print(f"  Column order preserved: {df_right.columns[2:7].tolist()}")

print(f"\n{'='*60}")
print(f"SPLITTING COMPLETE!")
print(f"{'='*60}")

REGION ORDER VERIFICATION
Total unique regions in df_transformed: 232
Expected: 232

First 10 regions:
  1. LH_VisCent_ExStr_2
  2. LH_VisCent_ExStr_1
  3. LH_VisCent_Striate_1
  4. LH_VisCent_ExStr_3
  5. LH_VisCent_ExStr_4
  6. LH_VisCent_ExStr_5
  7. LH_VisPeri_ExStrInf_1
  8. LH_VisPeri_ExStrInf_2
  9. LH_VisPeri_ExStrInf_3
  10. LH_VisPeri_StriCal_1

Last 10 regions:
  223. THA-VA-lh
  224. THA-DA-lh
  225. NAc-shell-lh
  226. NAc-core-lh
  227. pGP-lh
  228. aGP-lh
  229. aPUT-lh
  230. pPUT-lh
  231. aCAU-lh
  232. pCAU-lh

ORDER VERIFICATION
Expected first 5: ['LH_VisCent_ExStr_2', 'LH_VisCent_ExStr_1', 'LH_VisCent_Striate_1', 'LH_VisCent_ExStr_3', 'LH_VisCent_ExStr_4']
Actual first 5:   ['LH_VisCent_ExStr_2', 'LH_VisCent_ExStr_1', 'LH_VisCent_Striate_1', 'LH_VisCent_ExStr_3', 'LH_VisCent_ExStr_4']
Match: True

Expected last 5: ['aGP-lh', 'aPUT-lh', 'pPUT-lh', 'aCAU-lh', 'pCAU-lh']
Actual last 5:   ['aGP-lh', 'aPUT-lh', 'pPUT-lh', 'aCAU-lh', 'pCAU-lh']
Match: True

HEMISPHERE R

In [5]:
import pandas as pd
import numpy as np

# Step 1: Get the ordered list of regions for each hemisphere
print(f"{'='*60}")
print(f"REGION ORDER EXTRACTION")
print(f"{'='*60}")

# Extract unique regions in order from df_transformed
regions_ordered = df_transformed['region'].unique().tolist()

# Separate left and right hemisphere regions while maintaining order
left_regions_ordered = [reg for reg in regions_ordered if reg.startswith('LH_') or reg.endswith('-lh')]
right_regions_ordered = [reg for reg in regions_ordered if reg.startswith('RH_') or reg.endswith('-rh')]

print(f"Left hemisphere regions: {len(left_regions_ordered)}")
print(f"First 5: {left_regions_ordered[:5]}")
print(f"Last 5: {left_regions_ordered[-5:]}")

print(f"\nRight hemisphere regions: {len(right_regions_ordered)}")
print(f"First 5: {right_regions_ordered[:5]}")
print(f"Last 5: {right_regions_ordered[-5:]}")

# Step 2: Create upper triangle column names (i < j, no diagonal)
def create_upper_triangle_columns(regions):
    """Create upper triangle connectivity column names preserving region order"""
    columns = []
    n = len(regions)
    for i in range(n):
        for j in range(i + 1, n):  # i < j ensures upper triangle
            columns.append(f"{regions[i]}~{regions[j]}")
    return columns

left_upper_cols = create_upper_triangle_columns(left_regions_ordered)
right_upper_cols = create_upper_triangle_columns(right_regions_ordered)

print(f"\n{'='*60}")
print(f"UPPER TRIANGLE COLUMN GENERATION")
print(f"{'='*60}")
print(f"Left hemisphere connections: {len(left_upper_cols)}")
print(f"Expected: {116 * 115 // 2} = 6670")
print(f"First 5 connections: {left_upper_cols[:5]}")

print(f"\nRight hemisphere connections: {len(right_upper_cols)}")
print(f"Expected: {116 * 115 // 2} = 6670")
print(f"First 5 connections: {right_upper_cols[:5]}")

# Step 3: Transform left hemisphere to wide format
print(f"\n{'='*60}")
print(f"TRANSFORMING LEFT HEMISPHERE TO WIDE FORMAT")
print(f"{'='*60}")

# Create a dictionary to store the wide format data
left_wide_data = {'subject': df_left['subject'].unique()}

# For each upper triangle connection
for conn in left_upper_cols:
    region_i, region_j = conn.split('~')
    
    # Get connectivity values for this pair
    values = []
    for subject in left_wide_data['subject']:
        # Get the row where subject matches and region matches region_i
        row = df_left[(df_left['subject'] == subject) & (df_left['region'] == region_i)]
        if not row.empty:
            # Get the connectivity value from region_i to region_j
            value = row[region_j].values[0]
            values.append(value)
        else:
            values.append(np.nan)
    
    left_wide_data[conn] = values

df_left_wide = pd.DataFrame(left_wide_data)

print(f"Shape: {df_left_wide.shape}")
print(f"Expected: (224, 6671) - 1 subject column + 6670 connections")
print(f"\nFirst few rows and columns:")
print(df_left_wide.iloc[:3, :5])

# Step 4: Transform right hemisphere to wide format
print(f"\n{'='*60}")
print(f"TRANSFORMING RIGHT HEMISPHERE TO WIDE FORMAT")
print(f"{'='*60}")

# Create a dictionary to store the wide format data
right_wide_data = {'subject': df_right['subject'].unique()}

# For each upper triangle connection
for conn in right_upper_cols:
    region_i, region_j = conn.split('~')
    
    # Get connectivity values for this pair
    values = []
    for subject in right_wide_data['subject']:
        # Get the row where subject matches and region matches region_i
        row = df_right[(df_right['subject'] == subject) & (df_right['region'] == region_i)]
        if not row.empty:
            # Get the connectivity value from region_i to region_j
            value = row[region_j].values[0]
            values.append(value)
        else:
            values.append(np.nan)
    
    right_wide_data[conn] = values

df_right_wide = pd.DataFrame(right_wide_data)

print(f"Shape: {df_right_wide.shape}")
print(f"Expected: (224, 6671) - 1 subject column + 6670 connections")
print(f"\nFirst few rows and columns:")
print(df_right_wide.iloc[:3, :5])

# Step 5: Verify no missing values
print(f"\n{'='*60}")
print(f"DATA QUALITY CHECK")
print(f"{'='*60}")
print(f"Left hemisphere missing values: {df_left_wide.isna().sum().sum()}")
print(f"Right hemisphere missing values: {df_right_wide.isna().sum().sum()}")

# Step 6: Save to CSV files
print(f"\n{'='*60}")
print(f"SAVING TO CSV FILES")
print(f"{'='*60}")

output_dir = "/home/sjoon/projects/brain_connectivity_classifier/data/processed/"
left_filename = "LH_PIOP2_RestingState.csv"
right_filename = "RH_PIOP2_RestingState.csv"

df_left_wide.to_csv(output_dir + left_filename, index=False)
print(f"✓ Saved: {output_dir + left_filename}")
print(f"  Shape: {df_left_wide.shape}")

df_right_wide.to_csv(output_dir + right_filename, index=False)
print(f"✓ Saved: {output_dir + right_filename}")
print(f"  Shape: {df_right_wide.shape}")

# Step 7: Final verification
print(f"\n{'='*60}")
print(f"FINAL VERIFICATION")
print(f"{'='*60}")
print(f"Left hemisphere file:")
print(f"  Subjects: {len(df_left_wide)}")
print(f"  Connections: {len(df_left_wide.columns) - 1}")
print(f"  First connection: {df_left_wide.columns[1]}")
print(f"  Last connection: {df_left_wide.columns[-1]}")

print(f"\nRight hemisphere file:")
print(f"  Subjects: {len(df_right_wide)}")
print(f"  Connections: {len(df_right_wide.columns) - 1}")
print(f"  First connection: {df_right_wide.columns[1]}")
print(f"  Last connection: {df_right_wide.columns[-1]}")

print(f"\n{'='*60}")
print(f"TRANSFORMATION AND SAVE COMPLETE!")
print(f"{'='*60}")

REGION ORDER EXTRACTION
Left hemisphere regions: 116
First 5: ['LH_VisCent_ExStr_2', 'LH_VisCent_ExStr_1', 'LH_VisCent_Striate_1', 'LH_VisCent_ExStr_3', 'LH_VisCent_ExStr_4']
Last 5: ['aGP-lh', 'aPUT-lh', 'pPUT-lh', 'aCAU-lh', 'pCAU-lh']

Right hemisphere regions: 116
First 5: ['RH_VisCent_ExStr_1', 'RH_VisCent_ExStr_2', 'RH_VisCent_Striate_1', 'RH_VisCent_ExStr_3', 'RH_VisCent_ExStr_4']
Last 5: ['aGP-rh', 'aPUT-rh', 'pPUT-rh', 'aCAU-rh', 'pCAU-rh']

UPPER TRIANGLE COLUMN GENERATION
Left hemisphere connections: 6670
Expected: 6670 = 6670
First 5 connections: ['LH_VisCent_ExStr_2~LH_VisCent_ExStr_1', 'LH_VisCent_ExStr_2~LH_VisCent_Striate_1', 'LH_VisCent_ExStr_2~LH_VisCent_ExStr_3', 'LH_VisCent_ExStr_2~LH_VisCent_ExStr_4', 'LH_VisCent_ExStr_2~LH_VisCent_ExStr_5']

Right hemisphere connections: 6670
Expected: 6670 = 6670
First 5 connections: ['RH_VisCent_ExStr_1~RH_VisCent_ExStr_2', 'RH_VisCent_ExStr_1~RH_VisCent_Striate_1', 'RH_VisCent_ExStr_1~RH_VisCent_ExStr_3', 'RH_VisCent_ExStr_1~RH